# Módulo 06: Arquitectura Lakehouse con Delta Lake

## 1. Del Data Lake Tradicional al Lakehouse

En los Data Lakes convencionales basados en archivos Parquet o CSV:
* **Falta de transacciones ACID:** Fallos a mitad de escritura dejan datos corruptos o lecturas sucias (*dirty reads*).
* **Sin mutaciones a nivel de fila:** Actualizar o borrar registros exige reescribir particiones o tablas enteras.
* **Sin auditoría de versiones:** Resulta complejo rastrear cambios históricos o revertir errores de ingesta.

### Delta Lake como Capa de Almacenamiento Transaccional
**Delta Lake** añade una capa de metadatos (**`_delta_log/`**) sobre archivos Parquet que permite:
1. **Transacciones ACID:** Garantiza atomicidad y aislamiento en lecturas y escrituras concurrentes.
2. **Mutaciones atómicas:** Habilita operaciones directas de `UPDATE`, `DELETE` y `MERGE INTO`.
3. **Time Travel:** Permite consultar versiones históricas por número de versión o fecha.

In [1]:
import os
import sys
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable
import pyspark.sql.functions as F

# Asegurar que el JVM pueda cargar hadoop.dll/winutils.exe (requerido por Delta en Windows
# para listar _delta_log); si HADOOP_HOME\bin no está en PATH, forPath()/history() fallan
# con UnsatisfiedLinkError: NativeIO$Windows.access0
hadoop_home = os.environ.get("HADOOP_HOME", r"C:\hadoop")
hadoop_bin = os.path.join(hadoop_home, "bin")
if os.path.isdir(hadoop_bin) and hadoop_bin not in os.environ.get("PATH", ""):
    os.environ["HADOOP_HOME"] = hadoop_home
    os.environ["PATH"] = hadoop_bin + os.pathsep + os.environ.get("PATH", "")

# Si existe una sesión previa activa, la detenemos
if "spark" in locals() and spark is not None:
    spark.stop()

# Configuración del builder con la propiedad añadida
builder = (
    SparkSession.builder
    .appName("06_Delta_Lakehouse")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.hadoop.io.native.lib.available", "false")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark

## 2. Ingesta y Creación de la Tabla Delta (Versión 0)

Al persistir los datos con `.format("delta")`:
1. Spark escribe los registros en formato Parquet comprimido.
2. Genera el registro transaccional en `_delta_log/` con el commit de confirmación `00000000000000000000.json`.
3. Este primer guardado establece el punto de partida (**Versión 0**) de la tabla.S

In [2]:
import sys
import shutil

sys.path.append(os.path.abspath(".."))
from src.olympics_pipeline import DEPORTISTAS_SCHEMA

# Ruta canónica normalizada
delta_path = os.path.abspath("../data/processed/delta_deportistas").replace("\\", "/")

# Limpiar ejecuciones previas si existen
if os.path.exists(delta_path):
    shutil.rmtree(delta_path)

# Lectura estricta con esquema DDL
df_origen = (
    spark.read
    .schema(DEPORTISTAS_SCHEMA)
    .option("header", "true")
    .csv("../data/raw/deportista.csv")
)

# Escritura inicial en formato Delta (Commit Versión 0)
(
    df_origen
    .write
    .format("delta")
    .mode("overwrite")
    .save(delta_path)
)

print(f"Tabla Delta persistida en: {delta_path}")

Tabla Delta persistida en: c:/Users/Ruben/Desktop/Ciencia de datos/curso_spark/data/processed/delta_deportistas


## 3. Mutaciones ACID a Nivel de Fila (`UPDATE`)

En lugar de reescribir toda la tabla o partición, instanciamos la clase `DeltaTable` apuntando a la ruta física y ejecutamos un `update()` atómico a nivel de fila.

In [3]:
# 1. Vincular la tabla Delta existente
delta_tabla = DeltaTable.forPath(spark, delta_path)

# 2. Consultar estado previo (deportista_id = 1)
print("Estado previo a la modificación:")
delta_tabla.toDF().filter(F.col("deportista_id") == 1).select("deportista_id", "nombre", "edad", "peso").show()

# 3. Transacción UPDATE atómica
delta_tabla.update(
    condition="deportista_id = 1",
    set={
        "peso": F.lit(86.5),
        "edad": F.lit(25)
    }
)

# 4. Consultar estado actualizado
print("Estado posterior a la modificación (Versión 1):")
delta_tabla.toDF().filter(F.col("deportista_id") == 1).select("deportista_id", "nombre", "edad", "peso").show()

Estado previo a la modificación:
+-------------+---------+----+----+
|deportista_id|   nombre|edad|peso|
+-------------+---------+----+----+
|            1|A Dijiang|  24|80.0|
+-------------+---------+----+----+

Estado posterior a la modificación (Versión 1):
+-------------+---------+----+----+
|deportista_id|   nombre|edad|peso|
+-------------+---------+----+----+
|            1|A Dijiang|  25|86.5|
+-------------+---------+----+----+



## 4. Auditoría de Transacciones y Time Travel

Cada operación sobre una tabla Delta genera un nuevo registro inmutable en `_delta_log`. 


El log transaccional permite auditar todas las operaciones ejecutadas con `.history()` y realizar consultas retrospectivas (*Time Travel*) utilizando el parámetro **`versionAsOf`** (por índice de versión) o **`timestampAsOf`** (por fecha y hora).




In [4]:
# 1. Historial de transacciones registradas
print("Historial transaccional:")
delta_tabla.history().select("version", "timestamp", "operation", "operationParameters").show(truncate=False)

# 2. Lectura retrospectiva con Time Travel
df_v0 = (
    spark.read
    .format("delta")
    .option("versionAsOf", 0)
    .load(delta_path)
)

df_v1 = (
    spark.read
    .format("delta")
    .option("versionAsOf", 1)
    .load(delta_path)
)

print("Versión 0 (Original):")
df_v0.filter(F.col("deportista_id") == 1).select("deportista_id", "nombre", "edad", "peso").show()

print("Versión 1 (Post-UPDATE):")
df_v1.filter(F.col("deportista_id") == 1).select("deportista_id", "nombre", "edad", "peso").show()

Historial transaccional:
+-------+-----------------------+---------+------------------------------------------+
|version|timestamp              |operation|operationParameters                       |
+-------+-----------------------+---------+------------------------------------------+
|1      |2026-09-06 23:36:50.005|UPDATE   |{predicate -> ["(deportista_id#364 = 1)"]}|
|0      |2026-09-06 23:36:17.705|WRITE    |{mode -> Overwrite, partitionBy -> []}    |
+-------+-----------------------+---------+------------------------------------------+

Versión 0 (Original):
+-------------+---------+----+----+
|deportista_id|   nombre|edad|peso|
+-------------+---------+----+----+
|            1|A Dijiang|  24|80.0|
+-------------+---------+----+----+

Versión 1 (Post-UPDATE):
+-------------+---------+----+----+
|deportista_id|   nombre|edad|peso|
+-------------+---------+----+----+
|            1|A Dijiang|  25|86.5|
+-------------+---------+----+----+



## 5. Reto Práctico: Eliminación Atómica (`DELETE`) y Validación Histórica

### Instrucciones del Ejercicio:
1. Aplica el método `.delete()` sobre `delta_tabla` eliminando a los atletas que pertenezcan a `equipo_id == 2`.
2. Esta mutación generará la **Versión 2** en el log transaccional.
3. Lee la Versión 1 y la Versión 2 mediante Time Travel para comparar el aislamiento de los datos.
4. Valida mediante aserciones formales que el atleta exista en la Versión 1 y haya desaparecido en la Versión 2.

In [5]:
# 1. Ejecución de DELETE atómico (Genera Versión 2)
delta_tabla.delete(condition="equipo_id = 2")

# 2. Cargar Versión 2 mediante Time Travel
df_v2 = (
    spark.read
    .format("delta")
    .option("versionAsOf", 2)
    .load(delta_path)
)

# 3. Comparación y conteo entre versiones
conteo_v1 = df_v1.filter(F.col("equipo_id") == 2).count()
conteo_v2 = df_v2.filter(F.col("equipo_id") == 2).count()

print(f"Atletas de equipo_id = 2 en Versión 1: {conteo_v1}")
print(f"Atletas de equipo_id = 2 en Versión 2: {conteo_v2}")

# 4. Aserciones formales de validación
assert conteo_v1 == 1, f"Fallo en v1: se esperaba 1 atleta, se obtuvieron {conteo_v1}"
assert conteo_v2 == 0, f"Fallo en v2: se esperaban 0 atletas, se obtuvieron {conteo_v2}"

print("¡Aserción aprobada! Cuaderno 06 completado exitosamente.")

Atletas de equipo_id = 2 en Versión 1: 1
Atletas de equipo_id = 2 en Versión 2: 0
¡Aserción aprobada! Cuaderno 06 completado exitosamente.
